# Experimento de Escala de Contexto Espacial com Mosaicos 3×3 (11×11, 21×21, 33×33, 55×55, 99×99)
Quarto arquivo da sequência experimental:
1. **Testes iniciais** -> definição de baseline e métricas densas (`macro_RMSE_px`, `RMSE_px`, `RMSE_bldg_px`).
2. **SMOTE / cGAN** -> mitigação de desbalanceamento nas classes altas.
3. **Busca de arquitetura (`cnn_iterative_experimentation`)** -> seleção do melhor modelo base (`BEST_SPEC`).
4. **Este notebook (`context_scale_experimentation-v2`)** -> constrói mosaicos 3×3 a partir dos patches 33×33 contíguos para formar entradas de até **99×99 px** (`990×990 m`) e avalia as escalas `11`, `21`, `33`, `55` e `99` px de forma pareada e sem vazamento.

## 1. Setup, GPU e Limpeza de Memória

In [ ]:
# Setup: paths and shared folds (cnn/cnn_setup.py). Run this notebook from the repository root or from cnn/notebooks.
import sys, pathlib
for _p in (pathlib.Path.cwd(), pathlib.Path.cwd().parent, pathlib.Path.cwd() / "cnn"):
    if (_p / "cnn_setup.py").exists():
        sys.path.insert(0, str(_p)); break
import cnn_setup as cs

In [ ]:
import ctypes, gc, glob, hashlib, json, os, shutil, zlib
from pathlib import Path
import numpy as np, pandas as pd, tensorflow as tf, mlflow
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

try:
    _LIBC = ctypes.CDLL("libc.so.6")
except Exception:
    _LIBC = None

gpus = tf.config.list_physical_devices("GPU")
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)

def free_gpu():
    keras.backend.clear_session()
    for _ in range(3):
        gc.collect()
    if _LIBC is not None:
        try:
            _LIBC.malloc_trim(0)
        except Exception:
            pass

print("TF:", tf.__version__, "| GPUs:", gpus)

## 2. Configuração, Carregamento do Melhor Modelo Base e MLflow

In [ ]:
TRAIN_DIR = str(cs.TRAIN_DIR)
TEST_DIR = str(cs.TEST_DIR)
PATTERN = "*.tfrecord.gz"
DIST_GLOB = str(cs.DIST_GLOB)

C_IN, C_OUT, BASE_S, NODATA = 48, 3, 33, -9999.0
MOSAIC_S = BASE_S * 3                                  # 99x99 px via mosaico 3x3
CONTEXT_SCALES = (11, 21, 33, 55, 99)                  # ímpares: preservam o pixel central exato
EVAL_CORE_PX = 11                                      # núcleo comum para comparação justa entre escalas
STRICT_SAME_FOLD_NEIGHBORS = True                      # garante que nenhum vizinho 3x3 vaze entre folds

K_FOLDS, OUTER_FOLD = 5, 0
INNER_VAL_FOLD = (OUTER_FOLD + 1) % K_FOLDS
EPOCHS, LR, DROPOUT = 100, 1e-3, 0.3
AUG_ROTS = (0, 1, 2, 3)

BATCH_BY_PX = {11: 128, 21: 64, 33: 64, 55: 32, 99: 16}
PRED_BATCH_BY_PX = {11: 128, 21: 64, 33: 64, 55: 32, 99: 16}

PAPER_BINS = [-np.inf, 2, 6, 15, 30, np.inf]
PAPER_NAMES = ["h_lt2", "h_2_6", "h_6_15", "h_15_30", "h_gt30"]

PREV_RUNS_DIR = str(cs.OUT / "runs_cnn_iterative")
BEST_SPEC = {"name": "u32_c2", "filters": [32, 64, 128], "convs": 2, "se": False, "tatt": False, "loss": "rmse"}
cfg_candidates = sorted(glob.glob(f"{PREV_RUNS_DIR}/*/config.json"), key=os.path.getmtime)
if cfg_candidates:
    with open(cfg_candidates[-1], "r", encoding="utf-8") as f:
        prev_cfg = json.load(f)
    if "best_spec" in prev_cfg:
        BEST_SPEC = prev_cfg["best_spec"]
        print(f"Arquitetura carregada de {cfg_candidates[-1]}: {BEST_SPEC}")
else:
    print("Aviso: config.json anterior não encontrado; usando fallback:", BEST_SPEC)

NOTEBOOK_NAME = "context_scale_experimentation"
WEIGHTS_DIR = NOTEBOOK_NAME
OUT_DIR = str(cs.OUT / "runs_context_scale")
MLFLOW_URI = f"sqlite:///{OUT_DIR}/mlflow.db"
MLFLOW_EXPERIMENT = "context_scale_mosaic_99"

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(WEIGHTS_DIR, exist_ok=True)

In [ ]:
CONFIG = {
    "train_dir": TRAIN_DIR,
    "mosaic_grid": "3x3_of_33px",
    "scales": list(CONTEXT_SCALES),
    "eval_core_px": EVAL_CORE_PX,
    "strict_same_fold_neighbors": STRICT_SAME_FOLD_NEIGHBORS,
    "best_spec": BEST_SPEC,
    "k_folds": K_FOLDS,
    "outer_fold": OUTER_FOLD,
    "inner_val_fold": INNER_VAL_FOLD,
    "epochs": EPOCHS,
    "lr": LR,
    "dropout": DROPOUT,
    "seed": SEED,
}
CONFIG_HASH = hashlib.md5(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()[:12]

mlflow.set_tracking_uri(MLFLOW_URI)
if mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT) is None:
    mlflow.create_experiment(MLFLOW_EXPERIMENT, artifact_location=Path(OUT_DIR, "mlartifacts").resolve().as_uri())
mlflow.set_experiment(MLFLOW_EXPERIMENT)
if mlflow.active_run():
    mlflow.end_run(status="KILLED")

mlflow.start_run(run_name="context_mosaic_pipeline")
PARENT_RUN_ID = mlflow.active_run().info.run_id
RUN_DIR = f"{OUT_DIR}/{PARENT_RUN_ID}"
for sub in ("models_ctx", "figures"):
    os.makedirs(f"{RUN_DIR}/{sub}", exist_ok=True)

def out(name):
    return f"{RUN_DIR}/{name}"

mlflow.set_tags({"config_hash": CONFIG_HASH, "best_arch": BEST_SPEC["name"]})
mlflow.log_params({k: str(v) for k, v in CONFIG.items()})
print("Run MLflow ativo:", PARENT_RUN_ID)

## 3. Leitura dos Patches 33×33 e Montagem dos Mosaicos 3×3 (99×99 px)

In [ ]:
import glob, tensorflow as tf, pandas as pd

files = sorted(glob.glob("patch_dataset_train/*.tfrecord.gz"))
raw = next(iter(tf.data.TFRecordDataset(files[:1], compression_type="GZIP").take(1)))
ex = tf.train.Example()
ex.ParseFromString(raw.numpy())
print("Features no TFRecord:", sorted(ex.features.feature.keys()))

csvs = sorted(glob.glob("data/*/sample_points_natural_dist.csv"))
if csvs:
    df_pts = pd.read_csv(csvs[0])
    print("Colunas no CSV:", list(df_pts.columns))
    print(df_pts.head(3))

In [ ]:
COORD_KEYS = ("x_m", "y_m")   # the parser below reads x_m and y_m from the TFRecords, so they must exist there
print("Chaves espaciais usadas para indexar a grade 3x3:", COORD_KEYS)

spec_tfr = {
    "x": tf.io.FixedLenFeature([C_IN * BASE_S * BASE_S], tf.float32),
    "y": tf.io.FixedLenFeature([C_OUT * BASE_S * BASE_S], tf.float32),
    "height_m": tf.io.FixedLenFeature([1], tf.float32),
    "block_x": tf.io.FixedLenFeature([1], tf.float32),
    "block_y": tf.io.FixedLenFeature([1], tf.float32),
    "x_m": tf.io.FixedLenFeature([1], tf.float32),
    "y_m": tf.io.FixedLenFeature([1], tf.float32),
    "sample_id": tf.io.FixedLenFeature([1], tf.string),
    "city": tf.io.FixedLenFeature([1], tf.string),
    "split": tf.io.FixedLenFeature([1], tf.string),
}

def parse_base_33(rec):
    e = tf.io.parse_single_example(rec, spec_tfr)
    x = tf.transpose(tf.reshape(e["x"], [C_IN, BASE_S, BASE_S]), [1, 2, 0])
    y = tf.transpose(tf.reshape(e["y"], [C_OUT, BASE_S, BASE_S]), [1, 2, 0])
    valid = tf.cast(y[..., 0] != NODATA, tf.float32)
    bldg_h = tf.where(y[..., 1] == NODATA, 0.0, tf.maximum(y[..., 1], 0.0))
    return (x, bldg_h, valid, e["height_m"][0], e["block_x"][0], e["block_y"][0],
            e[COORD_KEYS[0]][0], e[COORD_KEYS[1]][0], e["sample_id"][0], e["city"][0], e["split"][0])

def load_all_33(folder, pattern, only_split="train"):
    files = sorted(glob.glob(f"{folder}/{pattern}", recursive=True))
    ds = tf.data.TFRecordDataset(files, compression_type="GZIP", num_parallel_reads=tf.data.AUTOTUNE)
    ds = ds.map(parse_base_33, num_parallel_calls=tf.data.AUTOTUNE)
    if only_split is not None:
        ds = ds.filter(lambda *a: tf.equal(a[10], only_split))
    cols = [np.concatenate(c) for c in zip(*ds.batch(512).prefetch(1).as_numpy_iterator())]
    d = dict(zip(["x", "h_map", "mask_px", "h", "bx", "by", "cx", "cy", "sid", "city", "split"], cols))
    for k in ("sid", "city", "split"):
        d[k] = np.array([s.decode() for s in d[k]])
    _, keep = np.unique(d["sid"], return_index=True)
    return {k: v[np.sort(keep)] for k, v in d.items()}

fold_of = cs.fold_of            # folds from data/folds.csv, shared with the classical models

def build_3x3_mosaics(d, strict_same_fold=STRICT_SAME_FOLD_NEIGHBORS):
    folds = fold_of(d["city"], d["bx"], d["by"], K_FOLDS)
    mosaic_indices = []
    center_indices = []
    EXPECTED_STEP_M = BASE_S * 10.0  # 330.0 m

    for city in np.unique(d["city"]):
        idx_c = np.flatnonzero(d["city"] == city)
        cx_c = d["cx"][idx_c].astype("float64")
        cy_c = d["cy"][idx_c].astype("float64")

        # Estima o passo real da grade via vizinho mais próximo ortogonal
        ux = np.sort(np.unique(np.round(cx_c, 2)))
        uy = np.sort(np.unique(np.round(cy_c, 2)))
        dx_diffs = np.diff(ux); dx_diffs = dx_diffs[dx_diffs > 1.0]
        dy_diffs = np.diff(uy); dy_diffs = dy_diffs[dy_diffs > 1.0]

        if len(dx_diffs) == 0 or len(dy_diffs) == 0:
            print(f"  {city:<16}: sem pontos suficientes para estimar passo.")
            continue

        min_dx, min_dy = float(np.min(dx_diffs)), float(np.min(dy_diffs))
        step_x = EXPECTED_STEP_M if abs(min_dx - EXPECTED_STEP_M) < 5.0 else min_dx
        step_y = EXPECTED_STEP_M if abs(min_dy - EXPECTED_STEP_M) < 5.0 else min_dy

        # Aceita apenas pontos alinhados em múltiplos exatos do passo (tolerância 2 m)
        gx_f = (cx_c - cx_c.min()) / step_x
        gy_f = (cy_c - cy_c.min()) / step_y
        aligned = (np.abs(gx_f - np.round(gx_f)) < 0.02) & (np.abs(gy_f - np.round(gy_f)) < 0.02)

        grid_map = {}
        for local_i in np.flatnonzero(aligned):
            grid_map[(int(round(gx_f[local_i])), int(round(gy_f[local_i])))] = int(idx_c[local_i])

        # Verifica orientação Norte-Sul pelas bordas compartilhadas
        err_north_up, err_north_down, n_chk = 0.0, 0.0, 0
        for (gx, gy), i_mid in list(grid_map.items())[:200]:
            if (gx, gy + 1) in grid_map:
                i_up = grid_map[(gx, gy + 1)]
                v1 = (d["x"][i_mid, 0, :, 0] != NODATA) & (d["x"][i_up, -1, :, 0] != NODATA)
                v2 = (d["x"][i_mid, -1, :, 0] != NODATA) & (d["x"][i_up, 0, :, 0] != NODATA)
                if v1.any():
                    err_north_up += float(np.abs(d["x"][i_mid, 0, v1, 0] - d["x"][i_up, -1, v1, 0]).mean())
                if v2.any():
                    err_north_down += float(np.abs(d["x"][i_mid, -1, v2, 0] - d["x"][i_up, 0, v2, 0]).mean())
                n_chk += 1
        y_ascending = (err_north_up <= err_north_down) if n_chk > 0 else True
        dy_signs = [+1, 0, -1] if y_ascending else [-1, 0, +1]

        city_count = 0
        for (gx, gy), i_center in grid_map.items():
            patch_mat = np.empty((3, 3), dtype=np.int64)
            ok = True
            for r_i, dy_s in enumerate(dy_signs):
                for c_i, dx_s in enumerate([-1, 0, +1]):
                    nb = grid_map.get((gx + dx_s, gy + dy_s))
                    if nb is None or (strict_same_fold and folds[nb] != folds[i_center]):
                        ok = False
                        break
                    patch_mat[r_i, c_i] = nb
                if not ok:
                    break
            if ok:
                mosaic_indices.append(patch_mat)
                center_indices.append(i_center)
                city_count += 1
        print(f"  {city:<16}: n={len(idx_c):>5} | min_step=({min_dx:.1f}m, {min_dy:.1f}m) | "
              f"alinhados={aligned.sum():>5} | mosaicos 3x3 válidos={city_count}")

    assert len(mosaic_indices) > 0, (
        "Nenhum mosaico 3x3 contíguo de 330m encontrado: os pontos (x_m, y_m) são amostragem esparsa, "
        "não uma grade regular contígua de 330m. Para treinar 55x55 e 99x99 será necessário exportar os TFRecords com S=99."
    )

    mosaic_indices = np.stack(mosaic_indices, axis=0)
    center_indices = np.array(center_indices, dtype=np.int64)
    n_mos = len(center_indices)

    X_mos = np.empty((n_mos, MOSAIC_S, MOSAIC_S, C_IN), dtype="float32")
    H_mos = np.empty((n_mos, MOSAIC_S, MOSAIC_S), dtype="float32")
    M_mos = np.empty((n_mos, MOSAIC_S, MOSAIC_S), dtype="float32")

    for r_i in range(3):
        r0, r1 = r_i * BASE_S, (r_i + 1) * BASE_S
        for c_i in range(3):
            c0, c1 = c_i * BASE_S, (c_i + 1) * BASE_S
            idxs = mosaic_indices[:, r_i, c_i]
            X_mos[:, r0:r1, c0:c1, :] = d["x"][idxs]
            H_mos[:, r0:r1, c0:c1] = d["h_map"][idxs]
            M_mos[:, r0:r1, c0:c1] = d["mask_px"][idxs]

    return {
        "x": X_mos,
        "h_map": H_mos,
        "mask_px": M_mos,
        "h": d["h"][center_indices],
        "bx": d["bx"][center_indices],
        "by": d["by"][center_indices],
        "sid": d["sid"][center_indices],
        "city": d["city"][center_indices],
        "split": d["split"][center_indices],
    }

raw_33 = load_all_33(TRAIN_DIR, PATTERN, only_split="train")
print(f"Total de patches 33x33 carregados: {len(raw_33['sid'])}")
mosaic_99 = build_3x3_mosaics(raw_33, strict_same_fold=STRICT_SAME_FOLD_NEIGHBORS)
raw_33 = None
gc.collect()

print(f"Mosaicos 99x99 prontos: {len(mosaic_99['sid'])} amostras | shape X = {mosaic_99['x'].shape}")
mlflow.log_param("n_mosaics_99", len(mosaic_99["sid"]))

## 4. Funções de Pré-processamento, Modelo e Métricas

In [ ]:
VALID_CH = [s * 12 + b for s in range(4) for b in (0, 2)]

def subset(d, m):
    return {k: v[m] for k, v in d.items()}

def crop_center(a, n):
    c, h = a.shape[1] // 2, n // 2
    return a[:, c - h:c + h + 1, c - h:c + h + 1]

def channel_stats(X, chunk=512):
    n = np.zeros(X.shape[-1]); s = np.zeros(X.shape[-1]); s2 = np.zeros(X.shape[-1])
    for i in range(0, len(X), chunk):
        xb = X[i:i + chunk].astype("float64")
        v = xb != NODATA
        xb = np.where(v, xb, 0.0)
        n += v.sum((0, 1, 2)); s += xb.sum((0, 1, 2)); s2 += (xb ** 2).sum((0, 1, 2))
    mean = s / n.clip(1)
    var = s2 / n.clip(1) - mean ** 2
    return mean.astype("float32"), np.sqrt(var.clip(0)).clip(1e-6).astype("float32")

def prep(X, mean, std, chunk=512):
    n, c = len(X), X.shape[3]
    arr = np.empty((n, X.shape[1], X.shape[2], c + len(VALID_CH)), dtype="float32")
    for s in range(0, n, chunk):
        xb = X[s:s + chunk]
        valid = xb != NODATA
        e = s + len(xb)
        arr[s:e, ..., :c] = np.where(valid, (xb - mean) / std, 0.0)
        arr[s:e, ..., c:] = valid[..., VALID_CH]
    return arr

def pack(h_map, mask):
    return np.stack([np.log1p(h_map), mask], -1).astype("float32")

def make_plan(n_real, rots=AUG_ROTS):
    return np.concatenate([np.stack([np.zeros(n_real, np.int64), np.arange(n_real), np.full(n_real, k)], 1) for k in rots]).astype(np.int64)

def make_ds_dense(X, Y, plan, batch, training, seed=SEED):
    n, rng = len(plan), np.random.default_rng(seed)
    def gen():
        order = rng.permutation(n) if training else np.arange(n)
        for s in range(0, n, batch):
            p = plan[order[s:s + batch]]
            xs, ys = [], []
            for rt in np.unique(p[:, 2]):
                sel = p[p[:, 2] == rt, 1]
                xb, yb = X[sel], Y[sel]
                if rt:
                    xb, yb = np.rot90(xb, rt, axes=(1, 2)), np.rot90(yb, rt, axes=(1, 2))
                xs.append(np.ascontiguousarray(xb))
                ys.append(np.ascontiguousarray(yb))
            yield np.concatenate(xs), np.concatenate(ys)
    sig = (tf.TensorSpec((None, *X.shape[1:]), tf.float32), tf.TensorSpec((None, *Y.shape[1:]), tf.float32))
    return tf.data.Dataset.from_generator(gen, output_signature=sig).prefetch(2)

In [ ]:
register = getattr(getattr(keras, "saving", None), "register_keras_serializable", None) or keras.utils.register_keras_serializable

@register(package="cnn_iter")
class SeasonSplit(layers.Layer):
    def call(self, x):
        h, w = x.shape[1], x.shape[2]
        bands = tf.reshape(x[..., :48], (-1, h, w, 4, 12))
        valid = tf.reshape(x[..., 48:], (-1, h, w, 4, 2))
        s = tf.transpose(tf.concat([bands, valid], -1), (0, 3, 1, 2, 4))
        return tf.reshape(s, (-1, h, w, 14))
    def compute_output_shape(self, input_shape):
        return (None, input_shape[1], input_shape[2], 14)

@register(package="cnn_iter")
class SeasonFuse(layers.Layer):
    def __init__(self, n_seasons=4, **kwargs):
        super().__init__(**kwargs)
        self.n_seasons = n_seasons
    def call(self, inputs):
        f, a = inputs
        h, w, c = f.shape[1], f.shape[2], f.shape[3]
        f = tf.reshape(f, (-1, self.n_seasons, h, w, c))
        a = tf.reshape(a, (-1, self.n_seasons, h, w, 1))
        return tf.reduce_sum(f * tf.nn.softmax(a, axis=1), axis=1)
    def compute_output_shape(self, input_shape):
        return (None, *input_shape[0][1:])
    def get_config(self):
        return {**super().get_config(), "n_seasons": self.n_seasons}

def conv_block(x, f):
    x = layers.Conv2D(f, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    return layers.ReLU()(x)

def se_block(x, r=8):
    c = x.shape[-1]
    s = layers.GlobalAveragePooling2D(keepdims=True)(x)
    s = layers.Dense(max(c // r, 4), activation="relu")(s)
    s = layers.Dense(c, activation="sigmoid")(s)
    return layers.Multiply()([x, s])

def build_from_spec(spec, input_shape, dropout=DROPOUT):
    h, w, ch = input_shape
    filters = spec["filters"]
    convs = spec.get("convs", 2)
    se = spec.get("se", False)
    tatt = spec.get("tatt", False)
    depth = len(filters) - 1
    p = (-h) % (2 ** depth)
    inp = layers.Input(input_shape)
    if tatt:
        f = conv_block(SeasonSplit()(inp), filters[0])
        f = conv_block(f, filters[0])
        x = SeasonFuse(4)([f, layers.Conv2D(1, 1)(f)])
    else:
        x = inp
    if p:
        x = layers.ZeroPadding2D(((0, p), (0, p)))(x)
    skips = []
    for f in filters[:-1]:
        for _ in range(convs):
            x = conv_block(x, f)
        if se:
            x = se_block(x)
        skips.append(x)
        x = layers.MaxPool2D()(x)
    x = conv_block(x, filters[-1])
    if se:
        x = se_block(x)
    x = layers.Dropout(dropout)(x)
    for f, sk in zip(filters[-2::-1], skips[::-1]):
        x = layers.UpSampling2D()(x)
        x = layers.Concatenate()([x, sk])
        for _ in range(convs):
            x = conv_block(x, f)
        if se:
            x = se_block(x)
    if p:
        x = layers.Cropping2D(((0, p), (0, p)))(x)
    return keras.Model(inp, layers.Conv2D(1, 1, activation="relu")(x))

def masked_rmse_loss(y_pack, y_pred):
    y, m = y_pack[..., 0], y_pack[..., 1]
    sq = tf.square(y - tf.squeeze(y_pred, -1))
    return tf.sqrt(tf.reduce_sum(sq * m) / (tf.reduce_sum(m) + 1e-8) + 1e-8)

def masked_rmse_mix_loss(y_pack, y_pred):
    y, m = y_pack[..., 0], y_pack[..., 1]
    sq = tf.square(y - tf.squeeze(y_pred, -1))
    mb = m * tf.cast(y > 0.0, m.dtype)
    r_all = tf.sqrt(tf.reduce_sum(sq * m) / (tf.reduce_sum(m) + 1e-8) + 1e-8)
    r_bld = tf.sqrt(tf.reduce_sum(sq * mb) / (tf.reduce_sum(mb) + 1e-8) + 1e-8)
    return 0.5 * r_all + 0.5 * r_bld

def masked_mae_log(y_pack, y_pred):
    y, m = y_pack[..., 0], y_pack[..., 1]
    return tf.reduce_sum(tf.abs(y - tf.squeeze(y_pred, -1)) * m) / (tf.reduce_sum(m) + 1e-8)

def predict_map_m(m, X, batch):
    p = np.concatenate([m.predict_on_batch(X[s:s + batch]) for s in range(0, len(X), batch)], axis=0)
    return np.clip(np.expm1(np.asarray(p)[..., 0]), 0, None)

def _m1d(y, p):
    err = p - y
    return {"MAE": float(np.abs(err).mean()), "RMSE": float(np.sqrt((err ** 2).mean())), "bias": float(err.mean()),
            "R2": float(1.0 - (err ** 2).sum() / (((y - y.mean()) ** 2).sum() + 1e-8))}

def dense_metrics(h_true, h_pred, mask):
    valid = mask > 0.5
    y, p = h_true[valid].astype("float64"), h_pred[valid].astype("float64")
    bld = y > 0
    c = h_true.shape[1] // 2
    m_all = _m1d(y, p)
    m_bld = _m1d(y[bld], p[bld]) if bld.any() else {k: float("nan") for k in m_all}
    m_ctr = _m1d(h_true[:, c, c].astype("float64"), h_pred[:, c, c].astype("float64"))
    idx = np.digitize(y, PAPER_BINS[1:-1], right=True)
    cls_rmse = {}
    for k, name in enumerate(PAPER_NAMES):
        s = idx == k
        if s.any():
            e = p[s] - y[s]
            cls_rmse[f"RMSE_{name}"] = float(np.sqrt((e ** 2).mean()))
    return {"macro_RMSE_px": float(np.mean(list(cls_rmse.values()))), "RMSE_px": m_all["RMSE"], "MAE_px": m_all["MAE"],
            "bias_px": m_all["bias"], "R2_px": m_all["R2"], "RMSE_bldg_px": m_bld["RMSE"], "MAE_bldg_px": m_bld["MAE"],
            "RMSE_center_px": m_ctr["RMSE"], **cls_rmse}

## 5. Split Único e Treino Pareado nas Escalas `11`, `21`, `33`, `55` e `99` px

In [ ]:
folds = fold_of(mosaic_99["city"], mosaic_99["bx"], mosaic_99["by"], K_FOLDS)
tr_99 = subset(mosaic_99, ~np.isin(folds, [OUTER_FOLD, INNER_VAL_FOLD]))
va_99 = subset(mosaic_99, folds == INNER_VAL_FOLD)
ho_99 = subset(mosaic_99, folds == OUTER_FOLD)

print(f"Mosaicos 99x99 -> treino: {len(tr_99['sid'])} | val interna: {len(va_99['sid'])} | hold-out: {len(ho_99['sid'])}")
mlflow.log_params({"n_train_mosaic": len(tr_99["sid"]), "n_val_mosaic": len(va_99["sid"]), "n_holdout_mosaic": len(ho_99["sid"])})

scale_results = []

for px in CONTEXT_SCALES:
    key = f"ctx_px{px}_{BEST_SPEC['name']}"
    batch = BATCH_BY_PX.get(px, 16)
    pred_batch = PRED_BATCH_BY_PX.get(px, 16)
    print(f"\n=== Escala {px}x{px} px ({px*10}x{px*10} m) | batch={batch} ===")

    free_gpu()
    x_tr_raw = crop_center(tr_99["x"], px)
    x_va_raw = crop_center(va_99["x"], px)
    hv = crop_center(va_99["h_map"], px).astype("float32")
    mv = crop_center(va_99["mask_px"], px).astype("float32")

    mean, std = channel_stats(x_tr_raw)
    Xt, Yt = prep(x_tr_raw, mean, std), pack(crop_center(tr_99["h_map"], px), crop_center(tr_99["mask_px"], px))
    Xv, Yv = prep(x_va_raw, mean, std), pack(hv, mv)
    x_tr_raw = x_va_raw = None
    gc.collect()

    plan_t = make_plan(len(Xt))
    plan_v = np.stack([np.zeros(len(Xv), np.int64), np.arange(len(Xv)), np.zeros(len(Xv), np.int64)], 1)

    m = h = ds_t = ds_v = pm = None
    try:
        m = build_from_spec(BEST_SPEC, Xt.shape[1:])
        m.compile(
            optimizer=keras.optimizers.Adam(LR),
            metrics=[masked_mae_log],
            loss=masked_rmse_mix_loss if BEST_SPEC.get("loss") == "mix" else masked_rmse_loss,
        )
        ckpt_tmp = os.path.join(WEIGHTS_DIR, f"{key}_partial.keras")
        cbs = [
            keras.callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=6, min_lr=1e-5),
            keras.callbacks.ModelCheckpoint(ckpt_tmp, monitor="val_loss", save_best_only=True),
            keras.callbacks.LambdaCallback(on_epoch_end=lambda ep, logs: mlflow.log_metrics({k: float(v) for k, v in logs.items()}, step=ep)),
        ]
        ds_t = make_ds_dense(Xt, Yt, plan_t, batch, True)
        ds_v = make_ds_dense(Xv, Yv, plan_v, batch, False)

        with mlflow.start_run(run_name=key, nested=True):
            mlflow.set_tags({"stage": "context_scale", "px": str(px)})
            mlflow.log_params({
                "px": px,
                "meters": px * 10,
                "arch": BEST_SPEC["name"],
                "batch": batch,
                "n_train": len(Xt),
                "n_val": len(Xv),
                "n_params": m.count_params(),
            })
            h = m.fit(ds_t, validation_data=ds_v, epochs=EPOCHS, callbacks=cbs, verbose=0)
            epochs_run = len(h.history["loss"])
            pm = predict_map_m(m, Xv, pred_batch)

            core_m = dense_metrics(crop_center(hv, EVAL_CORE_PX), crop_center(pm, EVAL_CORE_PX), crop_center(mv, EVAL_CORE_PX))
            full_m = dense_metrics(hv, pm, mv)

            mlflow.log_metrics({f"val_core_{k}": float(v) for k, v in core_m.items()})
            mlflow.log_metrics({f"val_full_{k}": float(v) for k, v in full_m.items()})
            mlflow.log_metric("epochs_run", epochs_run)

            weights_path = os.path.join(WEIGHTS_DIR, f"{key}.weights.h5")
            model_path = out(f"models_ctx/{key}.keras")
            norm_path = os.path.join(WEIGHTS_DIR, f"{key}_norm.npz")
            m.save_weights(weights_path)
            m.save(model_path)
            np.savez(norm_path, mean=mean, std=std, px=px)
            mlflow.log_artifact(weights_path, artifact_path="weights")
            if os.path.exists(ckpt_tmp):
                os.remove(ckpt_tmp)

            row = {
                "px": px,
                "area_m": f"{px*10}x{px*10}m",
                "epochs": epochs_run,
                "core11_macro_RMSE": core_m["macro_RMSE_px"],
                "core11_RMSE_px": core_m["RMSE_px"],
                "core11_RMSE_bldg": core_m["RMSE_bldg_px"],
                "core11_MAE_px": core_m["MAE_px"],
                "core11_R2": core_m["R2_px"],
                "full_macro_RMSE": full_m["macro_RMSE_px"],
                "full_RMSE_px": full_m["RMSE_px"],
                "full_RMSE_bldg": full_m["RMSE_bldg_px"],
                "model_path": model_path,
                "norm_path": norm_path,
            }
            scale_results.append(row)
            print(f"{key} | épocas={epochs_run} | core11 macro_RMSE={core_m['macro_RMSE_px']:.3f} m | "
                  f"core11 RMSE_bldg={core_m['RMSE_bldg_px']:.3f} m | full RMSE_px={full_m['RMSE_px']:.3f} m")
    except tf.errors.ResourceExhaustedError:
        print(f"{key}: sem memória na GPU mesmo com batch={batch}; reduza BATCH_BY_PX[{px}].")
    finally:
        if h is not None and hasattr(h, "history"):
            h.history.clear()
        del m, h, ds_t, ds_v, Xt, Yt, Xv, Yv, hv, mv, pm
        free_gpu()

df_scales = pd.DataFrame(scale_results)
df_scales.to_csv(out("context_scale_results.csv"), index=False)
mlflow.log_artifact(out("context_scale_results.csv"))
df_scales.drop(columns=["model_path", "norm_path"], errors="ignore").round(4)

## 6. Avaliação da Melhor Escala no Hold-Out (Fold Externo)

In [ ]:
assert len(df_scales) > 0, "Nenhuma escala foi concluída."
best_scale_row = df_scales.sort_values(["core11_macro_RMSE", "core11_RMSE_bldg"]).iloc[0]
BEST_PX = int(best_scale_row["px"])
BEST_CTX_MODEL_PATH = best_scale_row["model_path"]
BEST_NORM = np.load(best_scale_row["norm_path"])
print(f"Melhor escala selecionada: {BEST_PX}x{BEST_PX} px (core11 macro_RMSE = {best_scale_row['core11_macro_RMSE']:.3f} m)")

free_gpu()
X_ho = prep(crop_center(ho_99["x"], BEST_PX), BEST_NORM["mean"], BEST_NORM["std"])
h_map_ho = crop_center(ho_99["h_map"], BEST_PX).astype("float32")
mask_ho = crop_center(ho_99["mask_px"], BEST_PX).astype("float32")

best_ctx_model = keras.models.load_model(BEST_CTX_MODEL_PATH, compile=False)
p_ho = predict_map_m(best_ctx_model, X_ho, PRED_BATCH_BY_PX.get(BEST_PX, 16))

ho_core = dense_metrics(crop_center(h_map_ho, EVAL_CORE_PX), crop_center(p_ho, EVAL_CORE_PX), crop_center(mask_ho, EVAL_CORE_PX))
ho_full = dense_metrics(h_map_ho, p_ho, mask_ho)

mlflow.log_param("best_context_px", BEST_PX)
mlflow.log_metrics({f"ho_core11_{k}": float(v) for k, v in ho_core.items()})
mlflow.log_metrics({f"ho_full_{k}": float(v) for k, v in ho_full.items()})
mlflow.end_run()

pd.DataFrame({
    f"Hold-out núcleo {EVAL_CORE_PX}x{EVAL_CORE_PX}": ho_core,
    f"Hold-out mapa completo {BEST_PX}x{BEST_PX}": ho_full
}).T.round(3)